In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Download NLTK assets (only first time)
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:
!pip install -U transformers datasets accelerate
!pip install torchvision==0.22.1

  Using cached torchvision-0.22.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.1 kB)
  Using cached torch-2.7.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.6.77-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.6.77-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.6.80-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.5.1.17-py3-none-manylinux_2_28_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.6.4.1-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.3.0.4-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.7.77-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.7.1

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Text Cleaning

In [4]:
import pandas as pd

file_path = '/content/drive/MyDrive/Research_Project/Tickets_Data_20k.xlsx'
df = pd.read_excel(file_path)

df.head()

,Subject,Body,Answer,Type,Queue,Priority,Language,Tags
0,Unexpected crash of the data analytics platform,The data analysis platform crashed unexpectedl...,I will help you resolve the issue by restartin...,Incident,General Inquiry,low,en,"Crash,Technical,Bug,Hardware,Resolution,Outage..."
1,Customer Support Inquiry,Seeking information on digital strategies that...,We offer a variety of digital strategies and s...,Request,Customer Service,medium,en,"Feedback,Sales,IT,Tech Support"
2,Data Analytics for Investment,I am contacting you to request information on ...,I am here to assist you with data analytics to...,Request,Customer Service,medium,en,"Technical,Product,Guidance,Documentation,Perfo..."
3,Hospital service problem,Media data was locked due to unauthorized acce...,Returning to your email complaint about the at...,Incident,Customer Service,high,en,"Security,Breach,Login,Maintenance,Incident,Res..."
4,Security,"Dear Customer Support, I am reaching out to in...","Dear [name], we take the security of medical d...",Request,Customer Service,medium,en,"Security,Customer,Compliance,Breach,Documentat..."


In [5]:
df.shape

(48587, 8)

In [6]:
df['Subject'] = df['Subject'].fillna('')   # replace missing subject
df['text'] = df['Subject'] + " " + df['Body']

In [7]:
df = df[['text', 'Type', 'Tags', 'Queue', 'Priority', 'Answer']]
df = df.dropna()

In [8]:
print("After Selecting Columns:", df.shape)
df.head()

After Selecting Columns: (48574, 6)


,text,Type,Tags,Queue,Priority,Answer
0,Unexpected crash of the data analytics platfor...,Incident,"Crash,Technical,Bug,Hardware,Resolution,Outage...",General Inquiry,low,I will help you resolve the issue by restartin...
1,Customer Support Inquiry Seeking information o...,Request,"Feedback,Sales,IT,Tech Support",Customer Service,medium,We offer a variety of digital strategies and s...
2,Data Analytics for Investment I am contacting ...,Request,"Technical,Product,Guidance,Documentation,Perfo...",Customer Service,medium,I am here to assist you with data analytics to...
3,Hospital service problem Media data was locked...,Incident,"Security,Breach,Login,Maintenance,Incident,Res...",Customer Service,high,Returning to your email complaint about the at...
4,"Security Dear Customer Support, I am reaching ...",Request,"Security,Customer,Compliance,Breach,Documentat...",Customer Service,medium,"Dear [name], we take the security of medical d..."


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 48574 entries, 0 to 48586
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   text      48574 non-null  object
 1   Type      48574 non-null  object
 2   Tags      48574 non-null  object
 3   Queue     48574 non-null  object
 4   Priority  48574 non-null  object
 5   Answer    48574 non-null  object
dtypes: object(6)
memory usage: 2.6+ MB


In [10]:
text_cols = df.select_dtypes(include=['object']).columns

avg_words = df[text_cols].apply(lambda x: x.str.split().apply(len)).mean()
print("Average words per text column:")
print(avg_words)

Average words per text column:
text        59.913102
Type         1.000000
Tags         1.618520
Queue        2.259872
Priority     1.000000
Answer      58.465146
dtype: float64


In [11]:
stop_words = set(stopwords.words('english'))
lemma = WordNetLemmatizer()

In [12]:
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Initialize
stop_words = set(stopwords.words('english'))
lemma = WordNetLemmatizer()

def clean_text(text):
    # Convert to string
    text = str(text)
    # Convert to lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove placeholders/tags from ticket data
    text = re.sub(
        r'\b(name|customer_name|tel_num|email_id|ticket_id|userid|phone|address)\b',
        '',
        text
    )
    # Remove numbers (optional)
    text = re.sub(r'\d+', '', text)
    # Remove punctuation and special characters
    text = re.sub(
        '[' + re.escape(string.punctuation) + ']',
        ' ',
        text
    )
    # Remove single characters
    text = re.sub(r'\b[a-zA-Z]\b', '', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    # Tokenize
    words = text.split()
    # Remove stopwords + lemmatization
    words = [
        lemma.lemmatize(word)
        for word in words
        if word not in stop_words
    ]
    # Remove duplicate words while preserving order
    words = list(dict.fromkeys(words))

    return " ".join(words)

In [13]:
df['clean_text'] = df['text'].apply(clean_text)
df.head(10)

,text,Type,Tags,Queue,Priority,Answer,clean_text
0,Unexpected crash of the data analytics platfor...,Incident,"Crash,Technical,Bug,Hardware,Resolution,Outage...",General Inquiry,low,I will help you resolve the issue by restartin...,unexpected crash data analytics platform analy...
1,Customer Support Inquiry Seeking information o...,Request,"Feedback,Sales,IT,Tech Support",Customer Service,medium,We offer a variety of digital strategies and s...,customer support inquiry seeking information d...
2,Data Analytics for Investment I am contacting ...,Request,"Technical,Product,Guidance,Documentation,Perfo...",Customer Service,medium,I am here to assist you with data analytics to...,data analytics investment contacting request i...
3,Hospital service problem Media data was locked...,Incident,"Security,Breach,Login,Maintenance,Incident,Res...",Customer Service,high,Returning to your email complaint about the at...,hospital service problem medium data locked du...
4,"Security Dear Customer Support, I am reaching ...",Request,"Security,Customer,Compliance,Breach,Documentat...",Customer Service,medium,"Dear [name], we take the security of medical d...",security dear customer support reaching inquir...
5,Concerns About Securing Medical Data on 2-in-1...,Request,"Security,Product,Feature,IT,Tech Support,",Technical Support,medium,Thank you for your concern regarding securing ...,concern securing medical data convertible lapt...
6,Advice for backing up medical data in HubSpot ...,Request,"Backup,Security,IT,Tech Support",Technical Support,medium,We recommend backing up medical data in HubSpo...,advice backing medical data hubspot crm postgr...
7,Problem with Integration The integration stopp...,Problem,"Technical,Integration,Bug,Resolution,Outage,Do...",IT Support,high,I will look into the problem and call you at <...,problem integration stopped working unexpected...
8,"Assistance Request Dear Customer Support, I am...",Problem,"Technical,Bug,Security,Maintenance,Documentati...",Product Support,high,I have received your report about the data blo...,assistance request dear customer support writi...
9,Support Request The latest data analysis repor...,Problem,"Bug,Performance,IT,Tech Support",Product Support,high,Please provide additional details for further ...,support request latest data analysis report in...


#Preparing Data

In [14]:
!pip install transformers datasets scikit-learn pandas

In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from datasets import Dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments
import numpy as np

#Type Classification

In [16]:
df.columns

Index(['text', 'Type', 'Tags', 'Queue', 'Priority', 'Answer', 'clean_text'], dtype='object')

In [17]:
#Encoding labels
type_label_encoder = LabelEncoder()
df['label'] = type_label_encoder.fit_transform(df['Type'])

In [18]:
#Train-test split
train_df, test_df = train_test_split(df, test_size=0.40, random_state=42)

#Converting to HuggingFace Dataset

In [19]:
train_dataset = Dataset.from_pandas(train_df[['clean_text', 'label']])
test_dataset = Dataset.from_pandas(test_df[['clean_text', 'label']])

#Tokenization

In [20]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize(example):
    return tokenizer(
        example['clean_text'],
        truncation=True,
        padding='max_length',
        max_length=64   # faster
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/29144 [00:00<?, ? examples/s]

Map:   0%|          | 0/19430 [00:00<?, ? examples/s]

#Loading Model

In [21]:
type_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=len(type_label_encoder.classes_)
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


#Training Args

In [22]:
training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    logging_dir='./logs'
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [23]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average='weighted')
    }

In [24]:
trainer = Trainer(
    model=type_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [25]:
#train Model
trainer.train()

Step,Training Loss
500,0.631057
1000,0.466531
1500,0.455186
2000,0.410293
2500,0.372569
3000,0.385472
3500,0.357092
4000,0.331646
4500,0.285129
5000,0.290051


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=9110, training_loss=0.3089513527444113, metrics={'train_runtime': 1292.9374, 'train_samples_per_second': 112.705, 'train_steps_per_second': 7.046, 'total_flos': 2412979727585280.0, 'train_loss': 0.3089513527444113, 'epoch': 5.0})

In [26]:
trainer.evaluate()

Training Loss,Validation Loss,Step,Accuracy,F1
0.149753,0.500304,9110,0.848224,0.847774


{'eval_loss': 0.5003040432929993,
 'eval_accuracy': 0.848224395265054,
 'eval_f1': 0.8477737864188986}

In [27]:
import torch

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move model to GPU
type_model.to(device)

def predict_type(text):
    type_model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    )

    # 🔥 FIX: Move inputs to same device as model
    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = type_model(**inputs)

    preds = outputs.logits.argmax(dim=1).cpu().item()

    return type_label_encoder.inverse_transform([preds])[0]


# Test
print(predict_type("User unable to login to VPN"))

Problem


In [28]:
#Saving Model
type_model.save_pretrained("type_model")
tokenizer.save_pretrained("type_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('type_model/tokenizer_config.json', 'type_model/tokenizer.json')

In [29]:
#Saving Label Encoder
import pickle
with open("type_label_encoder.pkl", "wb") as f:
    pickle.dump(type_label_encoder, f)

#Queue Prediction

In [30]:
!pip install transformers datasets scikit-learn -q

In [31]:
import pandas as pd
import torch
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments
import pickle

In [32]:
#Encode Labels
queue_label_encoder = LabelEncoder()
df['label'] = queue_label_encoder.fit_transform(df['Queue'])

In [33]:
#Train-test split
train_df, test_df = train_test_split(df, test_size=0.40, random_state=42)

In [34]:
train_dataset = Dataset.from_pandas(train_df[['clean_text', 'label']])
test_dataset = Dataset.from_pandas(test_df[['clean_text', 'label']])

In [35]:
#Tokenization
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize(example):
    return tokenizer(
        example['clean_text'],
        truncation=True,
        padding='max_length',
        max_length=64   # faster
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

Map:   0%|          | 0/29144 [00:00<?, ? examples/s]

Map:   0%|          | 0/19430 [00:00<?, ? examples/s]

In [36]:
#Loading Model
queue_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=len(queue_label_encoder.classes_)
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [37]:
#Training Args
training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    logging_dir='./logs'
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [38]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average='weighted')
    }

In [39]:
trainer = Trainer(
    model=queue_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [40]:
trainer.train()

Step,Training Loss
500,1.853323
1000,1.717556
1500,1.657239
2000,1.607436
2500,1.574896
3000,1.549486
3500,1.523089
4000,1.455229
4500,1.409559
5000,1.412772


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=9110, training_loss=1.4146315841591057, metrics={'train_runtime': 1223.9293, 'train_samples_per_second': 119.059, 'train_steps_per_second': 7.443, 'total_flos': 2413237910784000.0, 'train_loss': 1.4146315841591057, 'epoch': 5.0})

In [41]:
trainer.evaluate()

Training Loss,Validation Loss,Step,Accuracy,F1
1.128076,1.455758,9110,0.500360,0.489811


{'eval_loss': 1.4557580947875977,
 'eval_accuracy': 0.5003602676273804,
 'eval_f1': 0.4898111947345693}

In [42]:
import torch

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move model to GPU
queue_model.to(device)

def predict_queue(text):
    queue_model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    )

    # 🔥 FIX: Move inputs to same device as model
    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = queue_model(**inputs)

    preds = outputs.logits.argmax(dim=1).cpu().item()

    return queue_label_encoder.inverse_transform([preds])[0]


# Test
print(predict_queue("User unable to login to VPN"))

Customer Service


In [43]:
#Saving Model
queue_model.save_pretrained("queue_model")
tokenizer.save_pretrained("queue_model")

with open("queue_label_encoder.pkl", "wb") as f:
    pickle.dump(queue_label_encoder, f)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [44]:
print(predict_queue("User unable to login after password reset"))

Technical Support


#Priority Prediction

In [45]:
!pip install transformers datasets scikit-learn -q

In [46]:
import pandas as pd
import torch
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments
import pickle

In [47]:
df.columns

Index(['text', 'Type', 'Tags', 'Queue', 'Priority', 'Answer', 'clean_text',
       'label'],
      dtype='object')

In [48]:
#Encoding labels
priority_label_encoder = LabelEncoder()
df['label'] = priority_label_encoder.fit_transform(df['Priority'])

In [49]:
#Train-test split
train_df, test_df = train_test_split(df, test_size=0.40, random_state=42)

#Converting to HuggingFace Dataset

In [50]:
train_dataset = Dataset.from_pandas(train_df[['clean_text', 'label']])
test_dataset = Dataset.from_pandas(test_df[['clean_text', 'label']])

#Tokenization

In [51]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize(example):
    return tokenizer(
        example['clean_text'],
        truncation=True,
        padding='max_length',
        max_length=64   # faster
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

Map:   0%|          | 0/29144 [00:00<?, ? examples/s]

Map:   0%|          | 0/19430 [00:00<?, ? examples/s]

#Loading Model

In [52]:
priority_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=len(priority_label_encoder.classes_)
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


#Training Args

In [53]:
training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    logging_dir='./logs'
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [54]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average='weighted')
    }

In [55]:
trainer = Trainer(
    model=priority_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [56]:
#train Model
trainer.train()

Step,Training Loss
500,1.059503
1000,1.042623
1500,1.029056
2000,1.004572
2500,0.977873
3000,0.964524
3500,0.951656
4000,0.885030
4500,0.859593
5000,0.832787


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=18220, training_loss=0.5707029346838741, metrics={'train_runtime': 2592.9816, 'train_samples_per_second': 112.396, 'train_steps_per_second': 7.027, 'total_flos': 4825873394104320.0, 'train_loss': 0.5707029346838741, 'epoch': 10.0})

In [57]:
trainer.evaluate()

Training Loss,Validation Loss,Step,Accuracy,F1
0.216599,1.347251,18220,0.648121,0.646342


{'eval_loss': 1.3472509384155273,
 'eval_accuracy': 0.6481214616572311,
 'eval_f1': 0.6463423716499155}

In [58]:
import torch

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move model to GPU
priority_model.to(device)

def predict_priority(text):
    priority_model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    )

    # 🔥 FIX: Move inputs to same device as model
    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = priority_model(**inputs)

    preds = outputs.logits.argmax(dim=1).cpu().item()

    return priority_label_encoder.inverse_transform([preds])[0]


# Test
print(predict_priority("User unable to login to VPN"))

medium


In [59]:
#Saving Model
priority_model.save_pretrained("priority_model")
tokenizer.save_pretrained("priority_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('priority_model/tokenizer_config.json', 'priority_model/tokenizer.json')

In [60]:
#Saving Label Encoder
import pickle
with open("priority_label_encoder.pkl", "wb") as f:
    pickle.dump(priority_label_encoder, f)

#Root Cause Analysis

In [61]:
!pip install transformers datasets -q

In [62]:
import pandas as pd
from datasets import Dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers import Trainer, TrainingArguments
import torch

In [63]:
#Loading and renaming columns for better understanding
df = df[['clean_text', 'Answer']].dropna()

df.columns = ['input', 'output']

In [64]:
df.columns

Index(['input', 'output'], dtype='object')

In [65]:
#Add instruction prefix
df['input'] = "generate root cause and solution: " + df['input']

In [66]:
#Converting to dataset
dataset = Dataset.from_pandas(df)

In [67]:
#Loading Tokenizer and Model
tokenizer = T5Tokenizer.from_pretrained("t5-small")
root_cause_model = T5ForConditionalGeneration.from_pretrained("t5-small")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [68]:
#Tokenization
def tokenize(example):
    inputs = tokenizer(
        example['input'],
        max_length=128,
        truncation=True,
        padding='max_length'
    )

    targets = tokenizer(
        example['output'],
        max_length=128,
        truncation=True,
        padding='max_length'
    )

    inputs['labels'] = targets['input_ids']
    return inputs

dataset = dataset.map(tokenize)

Map:   0%|          | 0/48574 [00:00<?, ? examples/s]

In [69]:
#Train-test split
dataset = dataset.train_test_split(test_size=0.1)

In [70]:
#Training
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"

from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./t5_results",
    per_device_train_batch_size=8,   # 🔥 FIXED
    per_device_eval_batch_size=8,
    num_train_epochs=2,              # keep small for speed
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=3000
)

trainer = Trainer(
    model=root_cause_model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test']
)

trainer.train()

Step,Training Loss
3000,1.423654
6000,1.165928
9000,1.112907


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=10930, training_loss=1.2092228197549748, metrics={'train_runtime': 1928.2201, 'train_samples_per_second': 45.343, 'train_steps_per_second': 5.668, 'total_flos': 2958301096574976.0, 'train_loss': 1.2092228197549748, 'epoch': 2.0})

In [71]:
#Saving Model
root_cause_model.save_pretrained("root_cause_model")
tokenizer.save_pretrained("root_cause_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('root_cause_model/tokenizer_config.json', 'root_cause_model/tokenizer.json')

In [72]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from transformers import T5Tokenizer, T5ForConditionalGeneration

tokenizer = T5Tokenizer.from_pretrained("root_cause_model")
model = T5ForConditionalGeneration.from_pretrained("root_cause_model").to(device)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [73]:
def predict_root_cause(text):
    model.eval()

    input_text = "generate root cause and solution: " + text

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    output_ids = model.generate(
        **inputs,
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

    result = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    return result

In [74]:
print(predict_root_cause("User unable to login after password reset"))

Dear name>, we apologize for the inconvenience caused by the password reset. To better assist you, could you please provide more details about the reset and the steps you have taken so far? We would like to schedule a call to discuss this further. Please let us know a suitable time for a call at tel_num>.


#Final Prediction

In [75]:
import pickle
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    T5Tokenizer,
    T5ForConditionalGeneration
)

# =====================================================
# DEVICE
# =====================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# =====================================================
# LOAD LABEL ENCODERS
# =====================================================

with open("type_label_encoder.pkl", "rb") as f:
    type_encoder = pickle.load(f)

with open("queue_label_encoder.pkl", "rb") as f:
    queue_encoder = pickle.load(f)

with open("priority_label_encoder.pkl", "rb") as f:
    priority_encoder = pickle.load(f)

# =====================================================
# LOAD MODELS
# =====================================================

# TYPE
type_tokenizer = AutoTokenizer.from_pretrained("type_model")
type_model = AutoModelForSequenceClassification.from_pretrained(
    "type_model"
).to(device)

# QUEUE
queue_tokenizer = AutoTokenizer.from_pretrained("queue_model")
queue_model = AutoModelForSequenceClassification.from_pretrained(
    "queue_model"
).to(device)

# PRIORITY
priority_tokenizer = AutoTokenizer.from_pretrained("priority_model")
priority_model = AutoModelForSequenceClassification.from_pretrained(
    "priority_model"
).to(device)

# ROOT CAUSE MODEL
root_tokenizer = T5Tokenizer.from_pretrained("root_cause_model")
root_model = T5ForConditionalGeneration.from_pretrained(
    "root_cause_model"
).to(device)

# =====================================================
# GENERIC CLASSIFICATION FUNCTION
# =====================================================

def classify(text, model, tokenizer, encoder):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(
        outputs.logits,
        dim=1
    ).item()

    confidence = torch.softmax(
        outputs.logits,
        dim=1
    )[0][prediction].item()

    label = encoder.inverse_transform(
        [prediction]
    )[0]

    return label, round(confidence * 100, 2)


# =====================================================
# ROOT CAUSE GENERATION
# =====================================================

def generate_root_cause(text):

    prompt = f"generate root cause and solution: {text}"

    inputs = root_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    root_model.eval()

    with torch.no_grad():

        output_ids = root_model.generate(
            **inputs,
            max_length=150,
            num_beams=4,
            early_stopping=True
        )

    result = root_tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return result


# =====================================================
# MASTER FUNCTION
# =====================================================

def predict_ticket(ticket_text):

    ticket_type, type_conf = classify(
        ticket_text,
        type_model,
        type_tokenizer,
        type_encoder
    )

    queue, queue_conf = classify(
        ticket_text,
        queue_model,
        queue_tokenizer,
        queue_encoder
    )

    priority, priority_conf = classify(
        ticket_text,
        priority_model,
        priority_tokenizer,
        priority_encoder
    )

    root_cause = generate_root_cause(ticket_text)

    return {
        "Type": ticket_type,
        "Type Confidence": type_conf,

        "Queue": queue,
        "Queue Confidence": queue_conf,

        "Priority": priority,
        "Priority Confidence": priority_conf,

        "Root Cause & Resolution": root_cause
    }


# TEST

if __name__ == "__main__":

    text = """
    User unable to login after password reset.
    VPN access is not working.
    """

    result = predict_ticket(text)

    print(result)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

{'Type': 'Problem', 'Type Confidence': 56.54, 'Queue': 'IT Support', 'Queue Confidence': 33.15, 'Priority': 'medium', 'Priority Confidence': 67.16, 'Root Cause & Resolution': 'name>, thank you for reaching out to us regarding the unable login after password reset. We apologize for the inconvenience caused. To better assist you, could you please provide more details about the password reset and the steps you have taken so far? We would like to schedule a call at your convenience to discuss this further. Please let us know a suitable time for the call at tel_num>.'}
